## Import Required Libraries
This notebook reuses the NASA battery data pipeline and evaluation protocol from the TE-Q-Transformer notebook, but replaces the model with a LSTM baseline for comparison.

In [1]:
!pip -q install pennylane scikit-learn seaborn matplotlib

import json
import os
import random
import shutil
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pennylane as qml
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from torch import nn, optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset


def seed_everything(seed: int = 42) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        if hasattr(torch.backends.cuda, 'enable_flash_sdp'):
            torch.backends.cuda.enable_flash_sdp(False)
        if hasattr(torch.backends.cuda, 'enable_mem_efficient_sdp'):
            torch.backends.cuda.enable_mem_efficient_sdp(False)
        if hasattr(torch.backends.cuda, 'enable_math_sdp'):
            torch.backends.cuda.enable_math_sdp(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)


seed_everything(42)

HYBRID_REQUIRED_CELLS = (
    'B0005', 'B0006', 'B0007',
    'B0018',
    'B0029', 'B0030', 'B0031', 'B0032',
    'B0053',
)
RAW_KAGGLE_HINT = '/kaggle/input/datasets/zadidallisan/quantum-sample/nasa'
REQUIRED_FILES = [
    f'{cell_id}_X.npy'
    for cell_id in HYBRID_REQUIRED_CELLS
] + [
    f'{cell_id}_soh.npy'
    for cell_id in HYBRID_REQUIRED_CELLS
]
OPTIONAL_FILES = [
    f'{cell_id}_X.npy'
    for cell_id in ('B0054', 'B0055', 'B0056')
] + [
    f'{cell_id}_soh.npy'
    for cell_id in ('B0054', 'B0055', 'B0056')
]
SAVE_DIR = Path('/kaggle/working/nasa_results/')
FIGURES_DIR = Path('/kaggle/working/nasa_figures/')
PREPROCESS_DIR = Path('/kaggle/working/nasa_preprocessed/')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESS_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is not enabled. Please switch the notebook accelerator to T4 GPU.')
DEVICE = torch.device('cuda')

warnings.filterwarnings(
    'ignore',
    message='enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True',
)


def _looks_like_nasa_dir(path: Path) -> bool:
    return path.exists() and path.is_dir() and all((path / name).exists() for name in REQUIRED_FILES)


def _candidate_paths_from_hint(raw_hint: str) -> List[Path]:
    hint = Path(raw_hint)
    candidates = [hint]

    parts = list(hint.parts)
    if 'datasets' in parts:
        idx = parts.index('datasets')
        if idx + 2 < len(parts):
            trimmed_parts = parts[:idx] + parts[idx + 2 :]
            candidates.append(Path(*trimmed_parts))
            if idx + 3 < len(parts):
                dataset_slug = parts[idx + 2]
                trailing = parts[idx + 3 :]
                candidates.append(Path('/kaggle/input') / dataset_slug / Path(*trailing))
                candidates.append(Path('/kaggle/input') / dataset_slug)

    candidates.extend([
        Path('/kaggle/input/quantum-sample/nasa'),
        Path('/kaggle/input/quantum-sample'),
    ])

    unique_candidates = []
    seen = set()
    for candidate in candidates:
        candidate_str = str(candidate)
        if candidate_str not in seen:
            unique_candidates.append(candidate)
            seen.add(candidate_str)
    return unique_candidates


def resolve_kaggle_data_dir(raw_hint: str) -> Path:
    for candidate in _candidate_paths_from_hint(raw_hint):
        if _looks_like_nasa_dir(candidate):
            print(f'Using matched dataset directory: {candidate}')
            return candidate

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        print('Scanning /kaggle/input recursively for the NASA NumPy files...')

        direct_dirs = [path for path in kaggle_input.rglob('*') if path.is_dir()]
        for directory in direct_dirs:
            if _looks_like_nasa_dir(directory):
                print(f'Auto-detected dataset directory: {directory}')
                return directory

        b0005_hits = list(kaggle_input.rglob('B0005_X.npy'))
        if b0005_hits:
            print('Found B0005_X.npy in these locations:')
            for hit in b0005_hits[:10]:
                print(f'  - {hit}')

    raise FileNotFoundError(
        'Could not locate the NASA NumPy dataset directory. '
        'Please verify the Kaggle dataset contents and ensure all 24 .npy files are present in one folder.'
    )


DATA_DIR = resolve_kaggle_data_dir(RAW_KAGGLE_HINT)

PRIMARY_BLUE = '#6F90AE'
ACCENT_GREEN = '#AFC8A7'
GRID_BLUE = '#C9D6E3'
TEXT_DARK = '#2F3B46'

plt.rcParams.update({
    'font.family': 'Times New Roman',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'axes.edgecolor': TEXT_DARK,
    'axes.labelcolor': TEXT_DARK,
    'xtick.color': TEXT_DARK,
    'ytick.color': TEXT_DARK,
    'text.color': TEXT_DARK,
})
sns.set_style('whitegrid')
sns.set_palette([PRIMARY_BLUE, ACCENT_GREEN])

print(f'Using device: {DEVICE}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'PREPROCESS_DIR: {PREPROCESS_DIR}')
print(f'SAVE_DIR: {SAVE_DIR}')
print(f'FIGURES_DIR: {FIGURES_DIR}')
print('Verified required files:')
for file_name in REQUIRED_FILES:
    print(f'  - {DATA_DIR / file_name}')

available_optional = [file_name for file_name in OPTIONAL_FILES if (DATA_DIR / file_name).exists()]
if available_optional:
    print('Optional files also found:')
    for file_name in available_optional:
        print(f'  - {DATA_DIR / file_name}')
else:
    print('Optional one-cycle files (B0054-B0056) are not present, which is acceptable for this hybrid run.')

print('Plot theme: light blue + light green reviewer palette enabled.')
print('Runtime note: PennyLane default.qubit is CPU-bound on Kaggle; the T4 mainly accelerates the PyTorch portions of the pipeline.')

BATCH_TO_CELLS = {
    0: ('B0005', 'B0006', 'B0007', 'B0018'),
    1: ('B0029', 'B0030', 'B0031', 'B0032'),
    2: ('B0053', 'B0054', 'B0055', 'B0056'),
}
def train_lstm_model(
    data_dir: Path = DATA_DIR,
    batch_size: int = 8,
    num_epochs: int = 80,
    patience: int = 20,
    learning_rate: float = 1e-3,
    weight_decay: float = 5e-2,
    scheduler_patience: int = 10,
    scheduler_factor: float = 0.5,
    grad_clip_norm: float = 1.0,
    save_root: Path = SAVE_DIR,
    model_cfg: Optional['LSTMVariant'] = None,
    experiment_name: str = 'lstm_baseline',
):
    seed_everything(42)
    save_dir = Path(save_root) / experiment_name
    save_dir.mkdir(parents=True, exist_ok=True)

    train_loader, test_loaders, _ = get_nasa_dataloaders(data_dir=data_dir, batch_size=batch_size, num_workers=0, pin_memory=True)
    model_cfg = model_cfg or LSTMVariant(name='baseline', d_model=64, hidden_size=64, n_layers=1)
    model = LSTMModel(
        d_model=model_cfg.d_model,
        hidden_size=model_cfg.hidden_size,
        n_layers=model_cfg.n_layers,
        bidirectional=model_cfg.bidirectional,
        dropout=model_cfg.dropout,
        head_hidden_dim=model_cfg.head_hidden_dim,
        use_projection=model_cfg.use_projection,
        pooling=model_cfg.pooling,
    ).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=scheduler_factor, patience=scheduler_patience)
    criterion = nn.MSELoss()

    history = {'epoch': [], 'train_loss': [], 'lr': [], 'epoch_seconds': []}
    training_config = {
        'experiment_name': experiment_name,
        'train_cells': list(FULL_TRAIN_CELLS),
        'test_cells': list(FULL_TEST_CELLS),
        'split_cell_id': SPLIT_CELL_ID,
        'split_rule': 'first 70% train, remaining 30% test',
        'batch_size': batch_size,
        'num_epochs': num_epochs,
        'patience': patience,
        'learning_rate': learning_rate,
        'weight_decay': weight_decay,
        'scheduler_patience': scheduler_patience,
        'scheduler_factor': scheduler_factor,
        'grad_clip_norm': grad_clip_norm,
        'device': str(DEVICE),
        'model_config': asdict(model_cfg),
    }
    _save_json(training_config, save_dir / 'lstm_training_config.json')

    best_train_loss = float('inf')
    best_epoch = 0
    patience_counter = 0

    print('Starting LSTM hybrid multi-temperature training.')
    print(f'Train samples: {len(train_loader.dataset)}')
    print('Per-cell test segments:', {cell_id: len(loader.dataset) for cell_id, loader in test_loaders.items()})

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.time()
        model.train()
        train_loss_sum = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
            optimizer.step()
            train_loss_sum += loss.item() * batch_x.size(0)

        train_loss = train_loss_sum / len(train_loader.dataset)
        scheduler.step(train_loss)
        current_lr = optimizer.param_groups[0]['lr']
        epoch_seconds = time.time() - epoch_start

        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['lr'].append(current_lr)
        history['epoch_seconds'].append(epoch_seconds)

        print(f"Epoch {epoch:03d}/{num_epochs} | Train Loss: {train_loss:.6f} | LR: {current_lr:.2e} | Time: {epoch_seconds:.1f}s")

        if train_loss < best_train_loss:
            best_train_loss = train_loss
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), save_dir / 'lstm_best.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping triggered after {patience} epochs without improvement.')
                break

    torch.save(model.state_dict(), save_dir / 'lstm_last.pth')
    _save_json(history, save_dir / 'lstm_training_history.json')
    _save_json({'best_epoch': best_epoch, 'best_train_loss': best_train_loss, 'epochs_completed': history['epoch'][-1] if history['epoch'] else 0}, save_dir / 'lstm_training_summary.json')
    return model, history, test_loaders, save_dir / 'lstm_best.pth', save_dir


print('LSTM notebook training cell ready.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 59.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 74.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 98.2 MB/s eta 0:00:00:00:0100:01
Scanning /kaggle/input recursively for the NASA NumPy files...
Auto-detected dataset directory: /kaggle/input/datasets/zadidallisan/li-ion-dataset/nasa/nasa
Using device: cuda
DATA_DIR: /kaggle/input/datasets/zadidallisan/li-ion-dataset/nasa/nasa
PREPROCESS_DIR: /kaggle/working/nasa_preprocessed
SAVE_DIR: /kaggle/working/nasa_results
FIGURES_DIR: /kaggle/working/nasa_figures
Verified required files:
  - /kaggle/i

In [2]:
def train_lstm_model(
    data_dir: Path = DATA_DIR,
    batch_size: int = 8,
    num_epochs: int = 80,
    patience: int = 20,
    learning_rate: float = 1e-3,
    weight_decay: float = 5e-2,
    scheduler_patience: int = 10,
    scheduler_factor: float = 0.5,
    grad_clip_norm: float = 1.0,
    save_root: Path = SAVE_DIR,
    model_cfg: Optional['LSTMVariant'] = None,
    experiment_name: str = 'lstm_baseline',
):
    seed_everything(42)
    save_dir = Path(save_root) / experiment_name
    save_dir.mkdir(parents=True, exist_ok=True)

    train_loader, test_loaders, _ = get_nasa_dataloaders(data_dir=data_dir, batch_size=batch_size, num_workers=0, pin_memory=True)
    model_cfg = model_cfg or LSTMVariant(name='baseline', d_model=64, hidden_size=64, n_layers=1)
    model = LSTMModel(
        d_model=model_cfg.d_model,
        hidden_size=model_cfg.hidden_size,
        n_layers=model_cfg.n_layers,
        bidirectional=model_cfg.bidirectional,
        dropout=model_cfg.dropout,
        head_hidden_dim=model_cfg.head_hidden_dim,
        use_projection=model_cfg.use_projection,
        pooling=model_cfg.pooling,
    ).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=scheduler_factor, patience=scheduler_patience)
    criterion = nn.MSELoss()

    history = {'epoch': [], 'train_loss': [], 'lr': [], 'epoch_seconds': []}
    training_config = {
        'experiment_name': experiment_name,
        'train_cells': list(FULL_TRAIN_CELLS),
        'test_cells': list(FULL_TEST_CELLS),
        'split_cell_id': SPLIT_CELL_ID,
        'split_rule': 'first 70% train, remaining 30% test',
        'batch_size': batch_size,
        'num_epochs': num_epochs,
        'patience': patience,
        'learning_rate': learning_rate,
        'weight_decay': weight_decay,
        'scheduler_patience': scheduler_patience,
        'scheduler_factor': scheduler_factor,
        'grad_clip_norm': grad_clip_norm,
        'device': str(DEVICE),
        'model_config': asdict(model_cfg),
    }
    _save_json(training_config, save_dir / 'lstm_training_config.json')

    best_train_loss = float('inf')
    best_epoch = 0
    patience_counter = 0

    print('Starting LSTM hybrid multi-temperature training.')
    print(f'Train samples: {len(train_loader.dataset)}')
    print('Per-cell test segments:', {cell_id: len(loader.dataset) for cell_id, loader in test_loaders.items()})

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.time()
        model.train()
        train_loss_sum = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
            optimizer.step()
            train_loss_sum += loss.item() * batch_x.size(0)

        train_loss = train_loss_sum / len(train_loader.dataset)
        scheduler.step(train_loss)
        current_lr = optimizer.param_groups[0]['lr']
        epoch_seconds = time.time() - epoch_start

        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['lr'].append(current_lr)
        history['epoch_seconds'].append(epoch_seconds)

        print(f"Epoch {epoch:03d}/{num_epochs} | Train Loss: {train_loss:.6f} | LR: {current_lr:.2e} | Time: {epoch_seconds:.1f}s")

        if train_loss < best_train_loss:
            best_train_loss = train_loss
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), save_dir / 'lstm_best.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping triggered after {patience} epochs without improvement.')
                break

    torch.save(model.state_dict(), save_dir / 'lstm_last.pth')
    _save_json(history, save_dir / 'lstm_training_history.json')
    _save_json({'best_epoch': best_epoch, 'best_train_loss': best_train_loss, 'epochs_completed': history['epoch'][-1] if history['epoch'] else 0}, save_dir / 'lstm_training_summary.json')
    return model, history, test_loaders, save_dir / 'lstm_best.pth', save_dir

## LSTM Dataset Wrapper
This wrapper matches the original notebook's shape checks so the dataloaders and evaluation code remain compatible.

In [3]:
class NASABatteryDataset(Dataset):
    def __init__(self, X: torch.Tensor, y: torch.Tensor) -> None:
        if X.ndim != 3 or X.shape[1:] != (512, 4):
            raise ValueError(f'Expected X shape [N, 512, 4], got {tuple(X.shape)}.')
        if y.ndim != 1:
            raise ValueError(f'Expected y shape [N], got {tuple(y.shape)}.')
        if len(X) != len(y):
            raise ValueError(f'X/y length mismatch: len(X)={len(X)}, len(y)={len(y)}.')
        self.X = X.float()
        self.y = y.float()

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]


FEATURE_IDX_TO_SCALE = (0, 1, 3)
FULL_TRAIN_CELLS = ('B0005', 'B0006', 'B0007', 'B0029', 'B0030', 'B0031')
FULL_TEST_CELLS = ('B0018', 'B0032')
SPLIT_CELL_ID = 'B0053'


def _load_cell_arrays(data_dir: Path, cell_id: str) -> Tuple[np.ndarray, np.ndarray]:
    x_path = data_dir / f'{cell_id}_X.npy'
    y_path = data_dir / f'{cell_id}_soh.npy'
    if not x_path.exists() or not y_path.exists():
        raise FileNotFoundError(f'Missing files for {cell_id} in {data_dir}.')

    X = np.load(x_path)
    y = np.load(y_path)

    if X.ndim != 3 or X.shape[1:] != (512, 4):
        raise ValueError(f'{cell_id}_X.npy must have shape [N, 512, 4], got {X.shape}.')
    if y.ndim != 1:
        raise ValueError(f'{cell_id}_soh.npy must have shape [N], got {y.shape}.')
    if len(X) != len(y):
        raise ValueError(f'{cell_id} length mismatch: len(X)={len(X)}, len(y)={len(y)}.')
    if len(y) == 0:
        raise ValueError(f'{cell_id} contains no cycles.')
    return X.astype(np.float32, copy=True), y.astype(np.float32, copy=False)


def _normalize_soh_per_cell(y: np.ndarray, cell_id: str) -> np.ndarray:
    c0 = float(y[0])
    if not np.isfinite(c0) or c0 == 0.0:
        raise ValueError(f'{cell_id} has invalid first-cycle capacity C0={c0}.')
    return (y / np.float32(c0)).astype(np.float32, copy=False)


def _load_full_cell(data_dir: Path, cell_id: str) -> Tuple[torch.Tensor, torch.Tensor]:
    X, y = _load_cell_arrays(data_dir, cell_id)
    y = _normalize_soh_per_cell(y, cell_id)
    return torch.from_numpy(X).float(), torch.from_numpy(y).float()


def _split_cell_first_second_half(data_dir: Path, cell_id: str) -> Tuple[Tuple[torch.Tensor, torch.Tensor], Tuple[torch.Tensor, torch.Tensor]]:
    X, y = _load_cell_arrays(data_dir, cell_id)
    y = _normalize_soh_per_cell(y, cell_id)
    split_idx = int(len(y) * 0.70)
    if split_idx == 0 or split_idx == len(y):
        raise ValueError(f'{cell_id} must contain enough cycles for a 70/30 split, got {len(y)}.')
    train_part = (torch.from_numpy(X[:split_idx].copy()).float(), torch.from_numpy(y[:split_idx].copy()).float())
    test_part = (torch.from_numpy(X[split_idx:].copy()).float(), torch.from_numpy(y[split_idx:].copy()).float())
    return train_part, test_part


def _fit_train_scaler(train_X: torch.Tensor) -> MinMaxScaler:
    scaler = MinMaxScaler(feature_range=(0.0, 1.0))
    flat_train = train_X.reshape(-1, train_X.shape[-1]).cpu().numpy()
    scaler.fit(flat_train[:, FEATURE_IDX_TO_SCALE])
    return scaler


def _apply_scaler(X: torch.Tensor, scaler: MinMaxScaler, clip_scaled: bool = True) -> torch.Tensor:
    X_np = X.cpu().numpy().astype(np.float32, copy=True)
    flat = X_np.reshape(-1, X_np.shape[-1])
    scaled = scaler.transform(flat[:, FEATURE_IDX_TO_SCALE])
    if clip_scaled:
        scaled = np.clip(scaled, 0.0, 1.0)
    flat[:, FEATURE_IDX_TO_SCALE] = scaled
    return torch.from_numpy(flat.reshape(X_np.shape)).float()


def get_nasa_dataloaders(
    data_dir: Path = DATA_DIR,
    batch_size: int = 32,
    num_workers: int = 0,
    pin_memory: bool = True,
):
    train_parts_X = []
    train_parts_y = []
    raw_test_parts = {}

    for cell_id in FULL_TRAIN_CELLS:
        X_cell, y_cell = _load_full_cell(Path(data_dir), cell_id)
        train_parts_X.append(X_cell)
        train_parts_y.append(y_cell)

    for cell_id in FULL_TEST_CELLS:
        X_cell, y_cell = _load_full_cell(Path(data_dir), cell_id)
        raw_test_parts[cell_id] = (X_cell, y_cell)

    (X_b0053_train, y_b0053_train), (X_b0053_test, y_b0053_test) = _split_cell_first_second_half(Path(data_dir), SPLIT_CELL_ID)
    train_parts_X.append(X_b0053_train)
    train_parts_y.append(y_b0053_train)
    raw_test_parts[f'{SPLIT_CELL_ID}_test'] = (X_b0053_test, y_b0053_test)

    X_train = torch.cat(train_parts_X, dim=0)
    y_train = torch.cat(train_parts_y, dim=0)

    scaler = _fit_train_scaler(X_train)
    X_train_scaled = _apply_scaler(X_train, scaler, clip_scaled=True)
    train_dataset = NASABatteryDataset(X_train_scaled, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)

    test_loaders = {}
    for cell_id, (X_test_cell, y_test_cell) in raw_test_parts.items():
        X_test_scaled = _apply_scaler(X_test_cell, scaler, clip_scaled=True)
        test_dataset = NASABatteryDataset(X_test_scaled, y_test_cell)
        test_loaders[cell_id] = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

    return train_loader, test_loaders, scaler

## Define LSTMModel
This section replaces the transformer architecture with a LSTM baseline while preserving the same input shape and regression target.

## Training Loop for LSTM
This section keeps the original optimization protocol: AdamW, MSE loss, ReduceLROnPlateau, gradient clipping, and early stopping with best-checkpoint saving.

In [ ]:
class LSTMModel(nn.Module):
    def __init__(
        self,
        d_model: int = 64,
        hidden_size: int = 64,
        n_layers: int = 1,
        bidirectional: bool = False,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
        use_projection: bool = True,
        pooling: str = 'last',
    ) -> None:
        super().__init__()
        if pooling not in {'last', 'mean'}:
            raise ValueError("pooling must be either 'last' or 'mean'.")
        self.d_model = d_model
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.bidirectional = bidirectional
        self.pooling = pooling
        self.use_projection = use_projection
        self.input_projection = nn.Linear(4, d_model) if use_projection else nn.Identity()
        lstm_dropout = dropout if n_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=d_model if use_projection else 4,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=lstm_dropout,
        )
        pool_dim = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.Linear(pool_dim, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3 or x.shape[-1] != 4:
            raise ValueError(f'Expected [B, L, 4], got {tuple(x.shape)}')
        x = self.input_projection(x)
        outputs, (h_n, _) = self.lstm(x)
        if self.pooling == 'mean':
            pooled = outputs.mean(dim=1)
        else:
            if self.bidirectional:
                forward_last = h_n[-2]
                backward_last = h_n[-1]
                pooled = torch.cat([forward_last, backward_last], dim=1)
            else:
                pooled = h_n[-1]
        return self.head(pooled).squeeze(-1)


def _load_lstm_model(variant: 'LSTMVariant', checkpoint_path: Path) -> LSTMModel:
    model = LSTMModel(
        d_model=variant.d_model,
        hidden_size=variant.hidden_size,
        n_layers=variant.n_layers,
        bidirectional=variant.bidirectional,
        dropout=variant.dropout,
        head_hidden_dim=variant.head_hidden_dim,
        use_projection=variant.use_projection,
        pooling=variant.pooling,
    ).to(DEVICE)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.eval()
    return model


def _save_json(payload: Dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2) + '\n')


def train_lstm_model(
    data_dir: Path = DATA_DIR,
    batch_size: int = 8,
    num_epochs: int = 80,
    patience: int = 20,
    learning_rate: float = 1e-3,
    weight_decay: float = 5e-2,
    scheduler_patience: int = 10,
    scheduler_factor: float = 0.5,
    grad_clip_norm: float = 1.0,
    save_root: Path = SAVE_DIR,
    model_cfg: Optional['LSTMVariant'] = None,
    experiment_name: str = 'lstm_baseline',
):
    seed_everything(42)
    save_dir = Path(save_root) / experiment_name
    save_dir.mkdir(parents=True, exist_ok=True)

    train_loader, test_loaders, _ = get_nasa_dataloaders(data_dir=data_dir, batch_size=batch_size, num_workers=0, pin_memory=True)
    model_cfg = model_cfg or LSTMVariant(name='baseline', d_model=64, hidden_size=64, n_layers=1)
    model = LSTMModel(
        d_model=model_cfg.d_model,
        hidden_size=model_cfg.hidden_size,
        n_layers=model_cfg.n_layers,
        bidirectional=model_cfg.bidirectional,
        dropout=model_cfg.dropout,
        head_hidden_dim=model_cfg.head_hidden_dim,
        use_projection=model_cfg.use_projection,
        pooling=model_cfg.pooling,
    ).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=scheduler_factor, patience=scheduler_patience)
    criterion = nn.MSELoss()

    history = {'epoch': [], 'train_loss': [], 'lr': [], 'epoch_seconds': []}
    training_config = {
        'experiment_name': experiment_name,
        'train_cells': list(FULL_TRAIN_CELLS),
        'test_cells': list(FULL_TEST_CELLS),
        'split_cell_id': SPLIT_CELL_ID,
        'split_rule': 'first 70% train, remaining 30% test',
        'batch_size': batch_size,
        'num_epochs': num_epochs,
        'patience': patience,
        'learning_rate': learning_rate,
        'weight_decay': weight_decay,
        'scheduler_patience': scheduler_patience,
        'scheduler_factor': scheduler_factor,
        'grad_clip_norm': grad_clip_norm,
        'device': str(DEVICE),
        'model_config': asdict(model_cfg),
    }
    _save_json(training_config, save_dir / 'lstm_training_config.json')

    best_train_loss = float('inf')
    best_epoch = 0
    patience_counter = 0

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.time()
        model.train()
        train_loss_sum = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
            optimizer.step()
            train_loss_sum += loss.item() * batch_x.size(0)

        train_loss = train_loss_sum / len(train_loader.dataset)
        scheduler.step(train_loss)
        current_lr = optimizer.param_groups[0]['lr']
        epoch_seconds = time.time() - epoch_start

        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['lr'].append(current_lr)
        history['epoch_seconds'].append(epoch_seconds)

        print(f"Epoch {epoch:03d}/{num_epochs} | Train Loss: {train_loss:.6f} | LR: {current_lr:.2e} | Time: {epoch_seconds:.1f}s")

        if train_loss < best_train_loss:
            best_train_loss = train_loss
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), save_dir / 'lstm_best.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping triggered after {patience} epochs without improvement.')
                break

    torch.save(model.state_dict(), save_dir / 'lstm_last.pth')
    _save_json(history, save_dir / 'lstm_training_history.json')
    _save_json({'best_epoch': best_epoch, 'best_train_loss': best_train_loss, 'epochs_completed': history['epoch'][-1] if history['epoch'] else 0}, save_dir / 'lstm_training_summary.json')
    return model, history, test_loaders, save_dir / 'lstm_best.pth', save_dir

## Run Inference and Evaluation
This section reuses the same metric computation, plots, and per-cell metric export layout as the reference notebook so LSTM results can be compared directly.

In [5]:
def compute_metrics(actual: np.ndarray, predicted: np.ndarray) -> Dict[str, float]:
    if actual.shape != predicted.shape:
        raise ValueError(f'Shape mismatch: actual {actual.shape}, predicted {predicted.shape}.')
    abs_error = np.abs(actual - predicted)
    denom = np.clip(np.abs(actual), a_min=1e-8, a_max=None)
    return {
        'RMSE': float(np.sqrt(mean_squared_error(actual, predicted))),
        'MAE': float(mean_absolute_error(actual, predicted)),
        'MAPE (%)': float(np.mean(abs_error / denom) * 100.0),
        'R2': float(r2_score(actual, predicted)),
        'MaxE': float(np.max(abs_error)),
    }


def plot_soh_trajectory(cycles: np.ndarray, actual: np.ndarray, predicted: np.ndarray, test_cell: str, output_dir: Path = FIGURES_DIR) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(7.0, 4.5))
    ax.plot(cycles, actual, color=PRIMARY_BLUE, linewidth=2.2, label='Actual SOH')
    ax.plot(cycles, predicted, color=ACCENT_GREEN, linestyle='--', linewidth=2.0, marker='o', markersize=3, label='Predicted SOH')
    ax.set_xlabel('Cycle Index')
    ax.set_ylabel('State of Health (SOH)')
    ax.set_title(f'LSTM Test Segment {test_cell}: Actual vs Predicted SOH')
    ax.legend(loc='best', frameon=True)
    ax.grid(True, linestyle='--', alpha=0.45, color=GRID_BLUE)
    plt.tight_layout()
    fig.savefig(output_dir / f'soh_trajectory_{test_cell}.pdf', bbox_inches='tight')
    fig.savefig(output_dir / f'soh_trajectory_{test_cell}.png', bbox_inches='tight')
    plt.close(fig)


def plot_error_violin(errors: np.ndarray, test_cell: str, output_dir: Path = FIGURES_DIR) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(5.0, 4.5))
    sns.violinplot(y=errors, ax=ax, inner='quart', color=ACCENT_GREEN, linewidth=1.2)
    ax.axhline(0.0, color=PRIMARY_BLUE, linestyle='--', linewidth=1.5, alpha=0.9)
    ax.set_ylabel('Prediction Error (Actual - Predicted)')
    ax.set_xlabel('')
    ax.set_title(f'Prediction Error Distribution ({test_cell})')
    ax.grid(True, linestyle='--', alpha=0.45, color=GRID_BLUE)
    plt.tight_layout()
    fig.savefig(output_dir / f'error_violin_{test_cell}.pdf', bbox_inches='tight')
    fig.savefig(output_dir / f'error_violin_{test_cell}.png', bbox_inches='tight')
    plt.close(fig)


def plot_parity(actual: np.ndarray, predicted: np.ndarray, dataset_name: str, output_dir: Path = FIGURES_DIR) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(5.0, 5.0))
    ax.scatter(actual, predicted, alpha=0.6, color=ACCENT_GREEN, edgecolor=PRIMARY_BLUE, s=20, label='Predictions')
    min_val = min(np.min(actual), np.min(predicted))
    max_val = max(np.max(actual), np.max(predicted))
    buffer = max((max_val - min_val) * 0.05, 1e-4)
    ax.plot([min_val - buffer, max_val + buffer], [min_val - buffer, max_val + buffer], color=TEXT_DARK, linestyle='--', linewidth=1.5, label='Perfect Prediction (y=x)')
    ax.set_xlim(min_val - buffer, max_val + buffer)
    ax.set_ylim(min_val - buffer, max_val + buffer)
    ax.set_xlabel('Actual SOH')
    ax.set_ylabel('Predicted SOH')
    ax.set_title(f'Parity Plot ({dataset_name})')
    ax.legend(loc='upper left', frameon=True)
    ax.grid(True, linestyle='--', alpha=0.45, color=GRID_BLUE)
    plt.tight_layout()
    fig.savefig(output_dir / f'parity_plot_{dataset_name}.pdf', bbox_inches='tight')
    fig.savefig(output_dir / f'parity_plot_{dataset_name}.png', bbox_inches='tight')
    plt.close(fig)


def save_metrics_outputs(metrics: Dict[str, float], test_cell: str, actual: np.ndarray, predicted: np.ndarray, output_dir: Path = SAVE_DIR) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / f'metrics_{test_cell}.txt').write_text('\n'.join(f'{k}: {v:.6f}' for k, v in metrics.items()) + '\n')
    (output_dir / f'metrics_{test_cell}.json').write_text(json.dumps(metrics, indent=2) + '\n')
    np.save(output_dir / f'actual_{test_cell}.npy', actual)
    np.save(output_dir / f'predicted_{test_cell}.npy', predicted)


def run_inference(model: nn.Module, data_loader: DataLoader, device: torch.device = DEVICE) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    actual_batches: List[np.ndarray] = []
    pred_batches: List[np.ndarray] = []
    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            pred_y = model(batch_x)
            actual_batches.append(batch_y.cpu().numpy())
            pred_batches.append(pred_y.cpu().numpy())
    return np.concatenate(actual_batches, axis=0), np.concatenate(pred_batches, axis=0)


def evaluate_model(model: nn.Module, test_loader: DataLoader, test_cell: str, figures_dir: Path, results_dir: Path) -> Dict[str, float]:
    actual, predicted = run_inference(model, test_loader, device=DEVICE)
    metrics = compute_metrics(actual, predicted)
    cycles = np.arange(1, len(actual) + 1)
    plot_soh_trajectory(cycles, actual, predicted, test_cell, figures_dir)
    plot_error_violin(actual - predicted, test_cell, figures_dir)
    plot_parity(actual, predicted, test_cell, figures_dir)
    save_metrics_outputs(metrics, test_cell, actual, predicted, results_dir)
    return metrics


@dataclass(frozen=True)
class LSTMVariant:
    name: str
    d_model: int
    hidden_size: int
    n_layers: int
    bidirectional: bool = False
    dropout: float = 0.0
    head_hidden_dim: int = 64
    use_projection: bool = True
    pooling: str = 'last'
    batch_size: int = 8
    num_epochs: int = 80
    patience: int = 20
    learning_rate: float = 1e-3
    weight_decay: float = 5e-2
    scheduler_factor: float = 0.5
    scheduler_patience: int = 10
    grad_clip_norm: float = 1.0


def build_LSTM_variants() -> List[LSTMVariant]:
    return [
        LSTMVariant(name='baseline', d_model=64, hidden_size=64, n_layers=1, bidirectional=False, dropout=0.0, head_hidden_dim=64, num_epochs=80),
        LSTMVariant(name='larger', d_model=64, hidden_size=128, n_layers=2, bidirectional=True, dropout=0.15, head_hidden_dim=128),
        LSTMVariant(name='smaller', d_model=32, hidden_size=32, n_layers=1, bidirectional=False, dropout=0.0, head_hidden_dim=32),
    ]


LSTM_VARIANTS = build_LSTM_variants()


def _load_teq_segment_arrays(save_root: Path, segment_id: str):
    matches = list(Path(save_root).rglob(f'actual_{segment_id}.npy'))
    if not matches:
        return None
    actual_path = matches[0]
    predicted_path = actual_path.with_name(f'predicted_{segment_id}.npy')
    if not predicted_path.exists():
        return None
    return np.load(actual_path), np.load(predicted_path)


def compare_lstm_to_teq(LSTM_summary: Dict[str, object], save_root: Path = SAVE_DIR) -> Dict[str, object]:
    teq_summary_path = Path(save_root) / 'nasa_hybrid_summary.json'
    teq_macro = None
    if teq_summary_path.exists():
        try:
            teq_summary = json.loads(teq_summary_path.read_text())
            teq_macro = teq_summary.get('macro_average_metrics')
        except Exception:
            teq_macro = None

    if teq_macro is None:
        segment_metrics = []
        for segment_id in LSTM_summary['test_cells']:
            loaded = _load_teq_segment_arrays(save_root, segment_id)
            if loaded is None:
                continue
            actual, predicted = loaded
            segment_metrics.append(compute_metrics(actual, predicted))
        if segment_metrics:
            teq_macro = {
                'RMSE': float(np.mean([m['RMSE'] for m in segment_metrics])),
                'MAE': float(np.mean([m['MAE'] for m in segment_metrics])),
                'MAPE (%)': float(np.mean([m['MAPE (%)'] for m in segment_metrics])),
                'R2': float(np.mean([m['R2'] for m in segment_metrics])),
                'MaxE': float(np.mean([m['MaxE'] for m in segment_metrics])),
            }

    comparison = {
        'LSTM_experiment': LSTM_summary['experiment_name'],
        'LSTM_macro_rmse': LSTM_summary['macro_rmse'],
        'LSTM_macro_mae': LSTM_summary['macro_mae'],
        'LSTM_macro_mape': LSTM_summary['macro_mape'],
        'LSTM_macro_r2': LSTM_summary['macro_r2'],
        'LSTM_macro_maxe': LSTM_summary['macro_maxe'],
        'teq_macro_metrics': teq_macro,
    }

    comparison_path = Path(LSTM_summary['run_dir']) / 'lstm_vs_teq_comparison.json'
    comparison_path.write_text(json.dumps(comparison, indent=2) + '\n')

    csv_lines = ['model,macro_rmse,macro_mae,macro_mape,macro_r2,macro_maxe']
    csv_lines.append(f"LSTM,{LSTM_summary['macro_rmse']:.6f},{LSTM_summary['macro_mae']:.6f},{LSTM_summary['macro_mape']:.6f},{LSTM_summary['macro_r2']:.6f},{LSTM_summary['macro_maxe']:.6f}")
    if teq_macro is not None:
        csv_lines.append(f"TEQTransformer,{teq_macro['RMSE']:.6f},{teq_macro['MAE']:.6f},{teq_macro['MAPE (%)']:.6f},{teq_macro['R2']:.6f},{teq_macro['MaxE']:.6f}")
    (Path(LSTM_summary['run_dir']) / 'lstm_vs_teq_comparison.csv').write_text('\n'.join(csv_lines) + '\n')
    return comparison

## Experiment Runner
This section trains the single LSTM baseline, evaluates it on the same held-out cells, and compares the macro metrics against any saved TE-Q-Transformer outputs.

In [6]:
LSTM_STUDY_SAVE_ROOT = SAVE_DIR / 'lstm_baseline'
RUN_LSTM_STUDY = True


@dataclass(frozen=True)
class LSTMVariant:
    name: str
    d_model: int
    hidden_size: int
    n_layers: int
    bidirectional: bool = False
    dropout: float = 0.0
    head_hidden_dim: int = 64
    use_projection: bool = True
    pooling: str = 'last'
    batch_size: int = 8
    num_epochs: int = 80
    patience: int = 20
    learning_rate: float = 1e-3
    weight_decay: float = 5e-2
    scheduler_factor: float = 0.5
    scheduler_patience: int = 10
    grad_clip_norm: float = 1.0


def build_LSTM_variants() -> List[LSTMVariant]:
    return [
        LSTMVariant(name='baseline', d_model=64, hidden_size=64, n_layers=1, bidirectional=False, dropout=0.0, head_hidden_dim=64),
    ]


LSTM_VARIANTS = build_LSTM_variants()


def _load_teq_segment_arrays(save_root: Path, segment_id: str):
    matches = list(Path(save_root).rglob(f'actual_{segment_id}.npy'))
    if not matches:
        return None
    actual_path = matches[0]
    predicted_path = actual_path.with_name(f'predicted_{segment_id}.npy')
    if not predicted_path.exists():
        return None
    return np.load(actual_path), np.load(predicted_path)


def compare_lstm_to_teq(LSTM_summary: Dict[str, object], save_root: Path = SAVE_DIR) -> Dict[str, object]:
    teq_summary_path = Path(save_root) / 'nasa_hybrid_summary.json'
    teq_macro = None
    if teq_summary_path.exists():
        try:
            teq_summary = json.loads(teq_summary_path.read_text())
            teq_macro = teq_summary.get('macro_average_metrics')
        except Exception:
            teq_macro = None

    if teq_macro is None:
        segment_metrics = []
        for segment_id in LSTM_summary['test_cells']:
            loaded = _load_teq_segment_arrays(save_root, segment_id)
            if loaded is None:
                continue
            actual, predicted = loaded
            segment_metrics.append(compute_metrics(actual, predicted))
        if segment_metrics:
            teq_macro = {
                'RMSE': float(np.mean([m['RMSE'] for m in segment_metrics])),
                'MAE': float(np.mean([m['MAE'] for m in segment_metrics])),
                'MAPE (%)': float(np.mean([m['MAPE (%)'] for m in segment_metrics])),
                'R2': float(np.mean([m['R2'] for m in segment_metrics])),
                'MaxE': float(np.mean([m['MaxE'] for m in segment_metrics])),
            }

    comparison = {
        'LSTM_experiment': LSTM_summary['experiment_name'],
        'LSTM_macro_rmse': LSTM_summary['macro_rmse'],
        'LSTM_macro_mae': LSTM_summary['macro_mae'],
        'LSTM_macro_mape': LSTM_summary['macro_mape'],
        'LSTM_macro_r2': LSTM_summary['macro_r2'],
        'LSTM_macro_maxe': LSTM_summary['macro_maxe'],
        'teq_macro_metrics': teq_macro,
    }

    comparison_path = Path(LSTM_summary['run_dir']) / 'lstm_vs_teq_comparison.json'
    comparison_path.write_text(json.dumps(comparison, indent=2) + '\n')

    csv_lines = ['model,macro_rmse,macro_mae,macro_mape,macro_r2,macro_maxe']
    csv_lines.append(
        f"LSTM,{LSTM_summary['macro_rmse']:.6f},{LSTM_summary['macro_mae']:.6f},{LSTM_summary['macro_mape']:.6f},{LSTM_summary['macro_r2']:.6f},{LSTM_summary['macro_maxe']:.6f}"
    )
    if teq_macro is not None:
        csv_lines.append(
            f"TEQTransformer,{teq_macro['RMSE']:.6f},{teq_macro['MAE']:.6f},{teq_macro['MAPE (%)']:.6f},{teq_macro['R2']:.6f},{teq_macro['MaxE']:.6f}"
        )
    (Path(LSTM_summary['run_dir']) / 'lstm_vs_teq_comparison.csv').write_text('\n'.join(csv_lines) + '\n')
    return comparison


def run_lstm_experiments(variants: Sequence[LSTMVariant] | None = None, save_root: Path = LSTM_STUDY_SAVE_ROOT) -> List[Dict[str, object]]:
    variants = list(variants or LSTM_VARIANTS)
    save_root = Path(save_root)
    save_root.mkdir(parents=True, exist_ok=True)
    summary_rows: List[Dict[str, object]] = []

    for index, variant in enumerate(variants, start=1):
        experiment_name = f'{index:02d}_{variant.name}'
        print(f'\n=== LSTM experiment {index:02d}/{len(variants)}: {experiment_name} ===')
        model, history, test_loaders, best_checkpoint_path, run_dir = train_lstm_model(
            data_dir=DATA_DIR,
            batch_size=variant.batch_size,
            num_epochs=variant.num_epochs,
            patience=variant.patience,
            learning_rate=variant.learning_rate,
            weight_decay=variant.weight_decay,
            scheduler_patience=variant.scheduler_patience,
            scheduler_factor=variant.scheduler_factor,
            grad_clip_norm=variant.grad_clip_norm,
            save_root=save_root,
            model_cfg=variant,
            experiment_name=experiment_name,
        )
        del model

        best_model = LSTMModel(
            d_model=variant.d_model,
            hidden_size=variant.hidden_size,
            n_layers=variant.n_layers,
            bidirectional=variant.bidirectional,
            dropout=variant.dropout,
            head_hidden_dim=variant.head_hidden_dim,
            use_projection=variant.use_projection,
            pooling=variant.pooling,
        ).to(DEVICE)
        best_model.load_state_dict(torch.load(best_checkpoint_path, map_location=DEVICE))
        best_model.eval()

        per_cell_metrics = {}
        for test_cell_id, test_loader in test_loaders.items():
            cell_figures_dir = run_dir / 'figures'
            cell_results_dir = run_dir / test_cell_id
            metrics = evaluate_model(best_model, test_loader, test_cell_id, cell_figures_dir, cell_results_dir)
            per_cell_metrics[test_cell_id] = metrics

        macro_metrics = {
            'RMSE': float(np.mean([m['RMSE'] for m in per_cell_metrics.values()])),
            'MAE': float(np.mean([m['MAE'] for m in per_cell_metrics.values()])),
            'MAPE (%)': float(np.mean([m['MAPE (%)'] for m in per_cell_metrics.values()])),
            'R2': float(np.mean([m['R2'] for m in per_cell_metrics.values()])),
            'MaxE': float(np.mean([m['MaxE'] for m in per_cell_metrics.values()])),
        }

        result_row = {
            'variant': variant.name,
            'experiment_name': experiment_name,
            'macro_rmse': macro_metrics['RMSE'],
            'macro_mae': macro_metrics['MAE'],
            'macro_mape': macro_metrics['MAPE (%)'],
            'macro_r2': macro_metrics['R2'],
            'macro_maxe': macro_metrics['MaxE'],
            'best_epoch': int(history['epoch'][-1]) if history['epoch'] else 0,
            'train_loss': float(min(history['train_loss'])) if history['train_loss'] else float('nan'),
            'test_cells': list(per_cell_metrics.keys()),
            'model_config': asdict(variant),
            'metrics': per_cell_metrics,
            'run_dir': str(run_dir),
        }
        summary_rows.append(result_row)
        _save_json(result_row, run_dir / 'lstm_result.json')

    summary_path = save_root / 'lstm_summary.json'
    summary_path.write_text(json.dumps(summary_rows, indent=2) + '\n')

    if summary_rows:
        best_variant = min(summary_rows, key=lambda row: row['macro_rmse'])
        leaderboard_lines = ['Leaderboard sorted by macro RMSE:']
        for row in sorted(summary_rows, key=lambda item: item['macro_rmse']):
            leaderboard_lines.append(
                f"- {row['experiment_name']}: RMSE={row['macro_rmse']:.6f}, MAE={row['macro_mae']:.6f}, R2={row['macro_r2']:.6f}"
            )
        leaderboard_lines.append('')
        leaderboard_lines.append(f"Best entry: {best_variant['experiment_name']} (macro RMSE={best_variant['macro_rmse']:.6f})")
        (save_root / 'lstm_summary.txt').write_text('\n'.join(leaderboard_lines) + '\n')
        compare_lstm_to_teq(best_variant, save_root=SAVE_DIR)
        print(f"\nBest LSTM entry by macro RMSE: {best_variant['experiment_name']}")

    return summary_rows


if RUN_LSTM_STUDY:
    lstm_results = run_lstm_experiments()
    print(f'Completed {len(lstm_results)} LSTM entries.')
else:
    print('LSTM experiment helpers are ready. Set RUN_LSTM_STUDY = True to train the baseline variants.')


=== LSTM experiment 01/1: 01_baseline ===
Epoch 001/80 | Train Loss: 0.023763 | LR: 1.00e-03 | Time: 1.7s
Epoch 002/80 | Train Loss: 0.012905 | LR: 1.00e-03 | Time: 0.3s
Epoch 003/80 | Train Loss: 0.011250 | LR: 1.00e-03 | Time: 0.3s
Epoch 004/80 | Train Loss: 0.011576 | LR: 1.00e-03 | Time: 0.3s
Epoch 005/80 | Train Loss: 0.011286 | LR: 1.00e-03 | Time: 0.3s
Epoch 006/80 | Train Loss: 0.011883 | LR: 1.00e-03 | Time: 0.3s
Epoch 007/80 | Train Loss: 0.010966 | LR: 1.00e-03 | Time: 0.3s
Epoch 008/80 | Train Loss: 0.012177 | LR: 1.00e-03 | Time: 0.3s
Epoch 009/80 | Train Loss: 0.011858 | LR: 1.00e-03 | Time: 0.3s
Epoch 010/80 | Train Loss: 0.012129 | LR: 1.00e-03 | Time: 0.3s
Epoch 011/80 | Train Loss: 0.011409 | LR: 1.00e-03 | Time: 0.3s
Epoch 012/80 | Train Loss: 0.010864 | LR: 1.00e-03 | Time: 0.3s
Epoch 013/80 | Train Loss: 0.011278 | LR: 1.00e-03 | Time: 0.3s
Epoch 014/80 | Train Loss: 0.011257 | LR: 1.00e-03 | Time: 0.3s
Epoch 015/80 | Train Loss: 0.010815 | LR: 1.00e-03 | Time: 0.

## Checkpointing and Export
This final section packages the LSTM experiment outputs into a single download bundle using the same save layout as the reference notebook.

In [7]:
bundle_dir = Path('/kaggle/working/NASA_lstm_baseline_Outputs')
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True, exist_ok=True)

shutil.copytree(SAVE_DIR, bundle_dir / 'lstm_results', dirs_exist_ok=True)
shutil.copytree(FIGURES_DIR, bundle_dir / 'lstm_figures', dirs_exist_ok=True)

best_model_path = SAVE_DIR / 'lstm_best.pth'
if best_model_path.exists():
    shutil.copy2(best_model_path, bundle_dir / best_model_path.name)

zip_path = shutil.make_archive('/kaggle/working/NASA_lstm_baseline_Outputs', 'zip', root_dir=bundle_dir)
print(f'Created zip archive: {zip_path}')

Created zip archive: /kaggle/working/NASA_lstm_baseline_Outputs.zip
